In [ ]:
import requests
from urllib.robotparser import RobotFileParser

url_robots = "https://www.portalinmobiliario.com/robots.txt"

agente = "UTFSM-ProyectoAcademico"

respuesta = requests.get(
    url_robots,
    headers={"User-Agent": agente},
    timeout=20
)

print("Código HTTP:", respuesta.status_code)

if respuesta.status_code == 200:
    robots = RobotFileParser()
    robots.parse(respuesta.text.splitlines())

    url_prueba = (
        "https://www.portalinmobiliario.com/"
        "departamento/santiago-metropolitana"
    )

    print(
        "Acceso según robots.txt:",
        robots.can_fetch(agente, url_prueba)
    )
else:
    print("No se pudieron verificar las reglas de robots.txt.")

In [ ]:
from bs4 import BeautifulSoup

url = (
    "https://www.portalinmobiliario.com/"
    "departamento/santiago-metropolitana"
)

respuesta = requests.get(
    url,
    headers={"User-Agent": agente},
    timeout=20
)

print("Código HTTP:", respuesta.status_code)
print("URL final:", respuesta.url)
print("Tamaño de respuesta:", len(respuesta.content), "bytes")

if respuesta.status_code == 200:
    soup = BeautifulSoup(respuesta.text, "html.parser")

    if soup.title:
        print("Título de la página:", soup.title.get_text(strip=True))
    else:
        print("No se encontró un título HTML.")

In [ ]:
# Buscar encabezados dentro del HTML
encabezados = soup.find_all(["h2", "h3"])

print("Cantidad de encabezados:", len(encabezados))

# Mostrar los primeros 10
for elemento in encabezados[:10]:
    print("-" * 50)
    print("Etiqueta:", elemento.name)
    print("Clase CSS:", elemento.get("class"))
    print("Texto:", elemento.get_text(" ", strip=True)[:150])

In [ ]:
# Obtener todos los enlaces de la página
enlaces = soup.find_all("a", href=True)

# Buscar enlaces que podrían corresponder a propiedades
enlaces_propiedades = [
    enlace for enlace in enlaces
    if "MLC-" in enlace["href"]
]

print("Total de enlaces:", len(enlaces))
print("Posibles enlaces de propiedades:", len(enlaces_propiedades))

# Examinar los primeros 5
for enlace in enlaces_propiedades[:5]:
    print("-" * 60)
    print("Texto:", enlace.get_text(" ", strip=True)[:120])
    print("Clase CSS:", enlace.get("class"))
    print("URL:", enlace["href"][:200])

In [ ]:
import re

# Eliminar duplicados usando el identificador MLC
publicaciones_unicas = {}

for enlace in enlaces_propiedades:
    url = enlace["href"]

    coincidencia = re.search(r"MLC-(\d+)", url)

    if coincidencia:
        id_publicacion = coincidencia.group(1)

        publicaciones_unicas[id_publicacion] = url.split("#")[0]

print("Enlaces encontrados:", len(enlaces_propiedades))
print("Publicaciones únicas:", len(publicaciones_unicas))

# Inspeccionar la estructura de la primera publicación
titulo = soup.select_one("a.poly-component__title[href*='MLC-']")

if titulo:
    for nivel, contenedor in enumerate(list(titulo.parents)[:7], 1):
        print(f"\n--- Nivel {nivel} ---")
        print("Etiqueta:", contenedor.name)
        print("Clases:", contenedor.get("class"))
        print("Texto:", contenedor.get_text(" ", strip=True)[:200])

In [ ]:
# Seleccionar el contenido de la primera publicación
tarjeta = soup.select_one(
    "li.ui-search-layout__item .poly-card__content"
)

if tarjeta:
    print("Contenido de la publicación:")
    print(tarjeta.get_text(" ", strip=True))

    print("\n--- ELEMENTOS RELEVANTES ---")

    palabras_clave = [
        "price",
        "money",
        "location",
        "attributes",
        "title"
    ]

    vistos = set()

    for elemento in tarjeta.find_all(True):
        clases = " ".join(elemento.get("class", []))
        texto = elemento.get_text(" ", strip=True)

        if any(p in clases.lower() for p in palabras_clave):

            if texto and len(texto) < 150:
                identificador = (clases, texto)

                if identificador not in vistos:
                    vistos.add(identificador)

                    print("-" * 45)
                    print("Etiqueta:", elemento.name)
                    print("Clase:", clases)
                    print("Texto:", texto)

                    if len(vistos) >= 20:
                        break
else:
    print("No se encontró el contenedor.")

In [ ]:
import re
from pprint import pprint

# Seleccionar la primera tarjeta inmobiliaria
tarjeta = soup.select_one("li.ui-search-layout__item")

def extraer_texto(elemento, selector):
    encontrado = elemento.select_one(selector)
    return encontrado.get_text(" ", strip=True) if encontrado else None

# Extraer información
titulo = extraer_texto(tarjeta, ".poly-component__title")
precio = extraer_texto(tarjeta, ".poly-component__price")
ubicacion = extraer_texto(tarjeta, ".poly-component__location")

atributos = [
    elemento.get_text(" ", strip=True)
    for elemento in tarjeta.select(".poly-attributes_list__item")
]

# Obtener enlace e identificador
enlace = tarjeta.select_one("a.poly-component__title")

url = enlace["href"].split("#")[0] if enlace else None

coincidencia = re.search(r"MLC-(\d+)", url or "")
id_publicacion = coincidencia.group(1) if coincidencia else None

# Identificar si corresponde a un proyecto
texto_tarjeta = tarjeta.get_text(" ", strip=True)

es_proyecto = texto_tarjeta.startswith("PROYECTO")

# Construir registro
publicacion = {
    "id": id_publicacion,
    "titulo": titulo,
    "precio_original": precio,
    "ubicacion": ubicacion,
    "atributos": atributos,
    "es_proyecto": es_proyecto,
    "url": url
}

pprint(publicacion, sort_dicts=False)

In [ ]:
import re
from pprint import pprint

# Obtener todas las tarjetas inmobiliarias
tarjetas = soup.select("li.ui-search-layout__item")

publicaciones = {}

for tarjeta in tarjetas:

    # Buscar el enlace principal
    enlace = tarjeta.select_one(
        "a.poly-component__title[href*='MLC-']"
    )

    if enlace is None:
        continue

    url = enlace["href"].split("#")[0]

    # Extraer el identificador
    coincidencia = re.search(r"MLC-(\d+)", url)

    if coincidencia is None:
        continue

    id_publicacion = coincidencia.group(1)

    # Extraer los datos
    titulo = extraer_texto(
        tarjeta, ".poly-component__title"
    )

    precio = extraer_texto(
        tarjeta, ".poly-component__price"
    )

    ubicacion = extraer_texto(
        tarjeta, ".poly-component__location"
    )

    atributos = [
        elemento.get_text(" ", strip=True)
        for elemento in tarjeta.select(
            ".poly-attributes_list__item"
        )
    ]

    texto_tarjeta = tarjeta.get_text(" ", strip=True)

    es_proyecto = texto_tarjeta.upper().startswith("PROYECTO")

    # Construir el registro
    publicaciones[id_publicacion] = {
        "id": id_publicacion,
        "titulo": titulo,
        "precio_original": precio,
        "ubicacion": ubicacion,
        "atributos": atributos,
        "es_proyecto": es_proyecto,
        "tipo_propiedad_busqueda": "departamento",
        "operacion_busqueda": "venta",
        "url": url
    }

# Convertir el diccionario en una lista
dataset = list(publicaciones.values())

print("Tarjetas encontradas:", len(tarjetas))
print("Publicaciones únicas extraídas:", len(dataset))

print("\nPrimeras 3 publicaciones:\n")

for publicacion in dataset[:3]:
    pprint(publicacion, sort_dicts=False)
    print("-" * 60)

In [ ]:
from collections import Counter

# Cantidad total de publicaciones
total = len(dataset)

# Clasificación de publicaciones
proyectos = sum(p["es_proyecto"] for p in dataset)
individuales = total - proyectos

# Comprobar valores faltantes
sin_titulo = sum(not p["titulo"] for p in dataset)
sin_precio = sum(not p["precio_original"] for p in dataset)
sin_ubicacion = sum(not p["ubicacion"] for p in dataset)

# Identificar precios mínimos de publicación
precios_desde = sum(
    bool(p["precio_original"]) and
    p["precio_original"].lower().startswith("desde")
    for p in dataset
)

# Comprobar duplicados
ids = [p["id"] for p in dataset]
duplicados = total - len(set(ids))

# Mostrar resumen
print("=== VALIDACIÓN DEL DATASET ===")
print(f"Total de publicaciones: {total}")
print(f"Proyectos inmobiliarios: {proyectos}")
print(f"Otras publicaciones: {individuales}")

print("\n=== VALORES FALTANTES ===")
print(f"Sin título: {sin_titulo}")
print(f"Sin precio: {sin_precio}")
print(f"Sin ubicación: {sin_ubicacion}")

print("\n=== PRECIOS Y DUPLICADOS ===")
print(f"Precios que comienzan con 'Desde': {precios_desde}")
print(f"Identificadores duplicados: {duplicados}")

In [ ]:
# Publicaciones no clasificadas como proyectos
otras_publicaciones = [
    p for p in dataset
    if not p["es_proyecto"]
]

print("Cantidad:", len(otras_publicaciones))

for i, p in enumerate(otras_publicaciones, 1):
    print(f"\n--- Publicación {i} ---")
    print("Título:", p["titulo"])
    print("Precio:", p["precio_original"])
    print("Ubicación:", p["ubicacion"])
    print("Atributos:", p["atributos"])
    print("URL:", p["url"])

# Ahora con filtro de Departamentos en venta usados en Santiago

In [ ]:
# URL de departamentos usados en venta
url_venta = (
    "https://www.portalinmobiliario.com/"
    "venta/departamento/propiedades-usadas/"
    "santiago-metropolitana/_FULL_true"
)

# Comprobar las reglas de acceso para esta nueva URL
permitido = robots.can_fetch(agente, url_venta)

print("Acceso permitido según robots.txt:", permitido)

if permitido:
    respuesta_venta = requests.get(
        url_venta,
        headers={"User-Agent": agente},
        timeout=20
    )

    print("Código HTTP:", respuesta_venta.status_code)

    if respuesta_venta.status_code == 200:
        soup_venta = BeautifulSoup(
            respuesta_venta.text,
            "html.parser"
        )

        tarjetas_venta = soup_venta.select(
            "li.ui-search-layout__item"
        )

        print("Tarjetas encontradas:", len(tarjetas_venta))

        # Examinar las primeras cinco publicaciones
        for tarjeta in tarjetas_venta[:5]:

            titulo = extraer_texto(
                tarjeta, ".poly-component__title"
            )

            precio = extraer_texto(
                tarjeta, ".poly-component__price"
            )

            print("-" * 50)
            print("Título:", titulo)
            print("Precio:", precio)
else:
    print("No continuaremos con la extracción de esta ruta.")

In [ ]:
# Seleccionar la segunda publicación
tarjeta = tarjetas_venta[1]

# Encontrar el contenedor del precio
bloque_precio = tarjeta.select_one(".poly-component__price")

print("=== ESTRUCTURA DEL PRECIO ===")

if bloque_precio:
    for elemento in bloque_precio.find_all(True):
        clases = " ".join(elemento.get("class", []))
        texto = elemento.get_text(" ", strip=True)

        if "price" in clases or "money" in clases:
            print("-" * 45)
            print("Etiqueta:", elemento.name)
            print("Clase:", clases)
            print("Texto:", texto[:120])
else:
    print("No se encontró el contenedor de precio.")

In [ ]:
# Probar la extracción de precios actuales y anteriores

for i, tarjeta in enumerate(tarjetas_venta[:5], 1):

    titulo = extraer_texto(
        tarjeta, ".poly-component__title"
    )

    precio_actual = extraer_texto(
        tarjeta,
        ".poly-price__current .poly-price__amount"
    )

    precio_anterior = extraer_texto(
        tarjeta,
        ".poly-price__previous"
    )

    print(f"\n--- Publicación {i} ---")
    print("Título:", titulo)
    print("Precio actual:", precio_actual)
    print("Precio anterior:", precio_anterior)

In [ ]:
import re
from datetime import datetime, timezone
from pprint import pprint

# Diccionario para evitar publicaciones duplicadas
publicaciones_venta = {}

# Fecha y hora de extracción
fecha_extraccion = datetime.now(timezone.utc).isoformat()

for tarjeta in tarjetas_venta:

    # Obtener enlace principal
    enlace = tarjeta.select_one(
        "a.poly-component__title[href*='MLC-']"
    )

    if enlace is None:
        continue

    url = enlace["href"].split("#")[0]

    # Obtener ID de la publicación
    coincidencia = re.search(r"MLC-(\d+)", url)

    if coincidencia is None:
        continue

    id_publicacion = coincidencia.group(1)

    # Extraer información
    titulo = extraer_texto(
        tarjeta, ".poly-component__title"
    )

    precio_actual = extraer_texto(
        tarjeta, ".poly-price__current .poly-price__amount"
    )

    precio_anterior = extraer_texto(
        tarjeta, ".poly-price__previous"
    )

    ubicacion = extraer_texto(
        tarjeta, ".poly-component__location"
    )

    atributos = [
        elemento.get_text(" ", strip=True)
        for elemento in tarjeta.select(
            ".poly-attributes_list__item"
        )
    ]

    # Construir registro
    publicaciones_venta[id_publicacion] = {
        "id": id_publicacion,
        "titulo": titulo,
        "precio_actual": precio_actual,
        "precio_anterior": precio_anterior,
        "ubicacion": ubicacion,
        "atributos": atributos,
        "tipo_propiedad_busqueda": "departamento",
        "operacion_busqueda": "venta",
        "modalidad_busqueda": "propiedades_usadas",
        "url": url,
        "fecha_extraccion": fecha_extraccion
    }

# Convertir a lista
dataset_venta = list(publicaciones_venta.values())

# Resumen
print("=== RESULTADOS DE EXTRACCIÓN ===")
print("Publicaciones extraídas:", len(dataset_venta))

print("Con precio actual:", sum(
    p["precio_actual"] is not None
    for p in dataset_venta
))

print("Con precio anterior:", sum(
    p["precio_anterior"] is not None
    for p in dataset_venta
))

print("Sin ubicación:", sum(
    p["ubicacion"] is None
    for p in dataset_venta
))

print("\n=== PRIMERAS 2 PUBLICACIONES ===")

for publicacion in dataset_venta[:2]:
    pprint(publicacion, sort_dicts=False)
    print("-" * 60)

In [ ]:
import json
from pathlib import Path

# Crear carpeta para los datos originales
carpeta = Path("data/raw/portal_inmobiliario")
carpeta.mkdir(parents=True, exist_ok=True)

# Nombre del archivo
ruta = carpeta / "portal_venta_departamentos_usados_santiago_p01.json"

# Guardar los registros
with open(ruta, "w", encoding="utf-8") as archivo:
    json.dump(
        dataset_venta,
        archivo,
        ensure_ascii=False,
        indent=2
    )

# Verificar el archivo guardado
with open(ruta, "r", encoding="utf-8") as archivo:
    datos_guardados = json.load(archivo)

print("Archivo guardado:", ruta.resolve())
print("Registros guardados:", len(datos_guardados))
print("Formato:", ruta.suffix)

In [ ]:
# Buscar elementos relacionados con la paginación
paginacion = soup_venta.select(
    "[class*='pagination']"
)

print("Elementos de paginación:", len(paginacion))

# Buscar enlaces dentro de esos elementos
enlaces_paginas = {}

for elemento in paginacion:
    for enlace in elemento.select("a[href]"):
        url = enlace.get("href")
        texto = enlace.get_text(" ", strip=True)

        enlaces_paginas[url] = texto

print("\n=== ENLACES DE PAGINACIÓN ===")

for url, texto in list(enlaces_paginas.items())[:15]:
    print("-" * 50)
    print("Texto:", texto)
    print("URL:", url)

In [ ]:
import re

# URL de la segunda página
url_pagina_2 = (
    "https://www.portalinmobiliario.com/"
    "venta/departamento/propiedades-usadas/"
    "santiago-metropolitana/"
    "_Desde_49_FULL_true_NoIndex_True"
)

# Verificar las reglas de acceso
permitido = robots.can_fetch(agente, url_pagina_2)

print("Acceso permitido:", permitido)

if permitido:
    respuesta_2 = requests.get(
        url_pagina_2,
        headers={"User-Agent": agente},
        timeout=20
    )

    print("Código HTTP:", respuesta_2.status_code)

    if respuesta_2.status_code == 200:

        soup_2 = BeautifulSoup(
            respuesta_2.text,
            "html.parser"
        )

        tarjetas_2 = soup_2.select(
            "li.ui-search-layout__item"
        )

        print("Tarjetas encontradas:", len(tarjetas_2))

        # Obtener los identificadores de la segunda página
        ids_pagina_2 = set()

        for tarjeta in tarjetas_2:
            enlace = tarjeta.select_one(
                "a.poly-component__title[href*='MLC-']"
            )

            if enlace:
                coincidencia = re.search(
                    r"MLC-(\d+)",
                    enlace.get("href", "")
                )

                if coincidencia:
                    ids_pagina_2.add(coincidencia.group(1))

        # Comparar con la primera página
        ids_pagina_1 = {
            p["id"] for p in dataset_venta
        }

        duplicados = ids_pagina_1 & ids_pagina_2

        print("Publicaciones únicas página 2:", len(ids_pagina_2))
        print("Repetidas entre ambas páginas:", len(duplicados))

        print("\nPrimeras 3 publicaciones de página 2:")

        for tarjeta in tarjetas_2[:3]:
            print("-" * 40)
            print(
                "Título:",
                extraer_texto(tarjeta, ".poly-component__title")
            )
            print(
                "Precio:",
                extraer_texto(
                    tarjeta,
                    ".poly-price__current .poly-price__amount"
                )
            )
else:
    print("No se realizará la solicitud.")

In [ ]:
def extraer_publicaciones(tarjetas, numero_pagina):

    resultados = {}

    fecha = datetime.now(timezone.utc).isoformat()

    for tarjeta in tarjetas:

        enlace = tarjeta.select_one(
            "a.poly-component__title[href*='MLC-']"
        )

        if enlace is None:
            continue

        url = enlace["href"].split("#")[0]

        coincidencia = re.search(r"MLC-(\d+)", url)

        if coincidencia is None:
            continue

        id_publicacion = coincidencia.group(1)

        atributos = [
            elemento.get_text(" ", strip=True)
            for elemento in tarjeta.select(
                ".poly-attributes_list__item"
            )
        ]

        resultados[id_publicacion] = {
            "id": id_publicacion,
            "titulo": extraer_texto(
                tarjeta, ".poly-component__title"
            ),
            "precio_actual": extraer_texto(
                tarjeta,
                ".poly-price__current .poly-price__amount"
            ),
            "precio_anterior": extraer_texto(
                tarjeta, ".poly-price__previous"
            ),
            "ubicacion": extraer_texto(
                tarjeta, ".poly-component__location"
            ),
            "atributos": atributos,
            "tipo_propiedad_busqueda": "departamento",
            "operacion_busqueda": "venta",
            "modalidad_busqueda": "propiedades_usadas",
            "pagina_resultados": numero_pagina,
            "url": url,
            "fecha_extraccion": fecha
        }

    return list(resultados.values())

In [ ]:
# Extraer los registros de la segunda página
datos_pagina_2 = extraer_publicaciones(
    tarjetas_2,
    numero_pagina=2
)

# Utilizar los registros ya extraídos de la primera página
datos_pagina_1 = [
    {**registro, "pagina_resultados": 1}
    for registro in dataset_venta
]

# Combinar ambas páginas evitando IDs duplicados
publicaciones_totales = {}

for registro in datos_pagina_1 + datos_pagina_2:
    publicaciones_totales.setdefault(
        registro["id"],
        registro
    )

dataset_acumulado = list(publicaciones_totales.values())

print("=== RESULTADOS ACUMULADOS ===")
print("Página 1:", len(datos_pagina_1))
print("Página 2:", len(datos_pagina_2))
print("Total de registros únicos:", len(dataset_acumulado))

print("\nRegistros por página:")

for pagina in [1, 2]:
    cantidad = sum(
        p["pagina_resultados"] == pagina
        for p in dataset_acumulado
    )
    print(f"Página {pagina}: {cantidad}")

In [ ]:
import time
import requests
from bs4 import BeautifulSoup

# Partimos de las 96 publicaciones que ya tenemos
registros = {
    p["id"]: p
    for p in dataset_acumulado
}

# Probar páginas 3 y 4
for pagina in range(3, 5):

    inicio = (pagina - 1) * 48 + 1

    url = (
        "https://www.portalinmobiliario.com/"
        "venta/departamento/propiedades-usadas/"
        f"santiago-metropolitana/_Desde_{inicio}"
        "_FULL_true_NoIndex_True"
    )

    print(f"\n=== PÁGINA {pagina} ===")

    # Verificar si la ruta permite acceso automatizado
    if not robots.can_fetch(agente, url):
        print("Acceso no permitido. Extracción detenida.")
        break

    # Pausa para limitar la frecuencia de solicitudes
    time.sleep(4)

    respuesta = requests.get(
        url,
        headers={"User-Agent": agente},
        timeout=25
    )

    print("Código HTTP:", respuesta.status_code)

    if respuesta.status_code != 200:
        print("No se pudo acceder. Extracción detenida.")
        break

    soup_pagina = BeautifulSoup(
        respuesta.text,
        "html.parser"
    )

    tarjetas = soup_pagina.select(
        "li.ui-search-layout__item"
    )

    if not tarjetas:
        print("No se encontraron publicaciones.")
        break

    datos = extraer_publicaciones(
        tarjetas,
        numero_pagina=pagina
    )

    nuevos = 0

    for publicacion in datos:
        if publicacion["id"] not in registros:
            registros[publicacion["id"]] = publicacion
            nuevos += 1

    print("Publicaciones encontradas:", len(datos))
    print("Publicaciones nuevas:", nuevos)

    if nuevos == 0:
        print("No hay registros nuevos. Extracción detenida.")
        break

# Dataset acumulado
dataset_acumulado = list(registros.values())

print("\n=== RESUMEN FINAL ===")
print("Total de publicaciones únicas:", len(dataset_acumulado))

In [ ]:
import json
from pathlib import Path
from datetime import datetime

# Ruta del repositorio clonado
repositorio = Path(
    r"C:\Users\felip\Documents\GitHub\Proyecto_inmobiliario"
)

# Carpeta de destino
carpeta = repositorio / "datalake_crudo" / "portal_inmobiliario"

# Crear carpeta si no existe
carpeta.mkdir(parents=True, exist_ok=True)

# Nombre del archivo con fecha y hora
fecha = datetime.now().strftime("%Y%m%d_%H%M%S")

archivo = carpeta / f"portal_venta_departamentos_santiago_{fecha}.json"

# Guardar las publicaciones acumuladas
with open(archivo, "w", encoding="utf-8") as f:
    json.dump(
        dataset_acumulado,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Archivo guardado en:", archivo)
print("Registros:", len(dataset_acumulado))

In [1]:
import json
from pathlib import Path

# Carpeta de los datos extraídos
carpeta = Path(
    r"C:\Users\felip\Documents\GitHub\Proyecto_inmobiliario"
) / "datalake_crudo" / "portal_inmobiliario"

# Buscar los archivos generados por el script
archivos = list(
    carpeta.glob("portal_venta_departamento_*.json")
)

if not archivos:
    print("No se encontraron archivos JSON del scraper.")

else:
    # Seleccionar el archivo más reciente
    archivo = max(
        archivos,
        key=lambda ruta: ruta.stat().st_mtime
    )

    with open(archivo, "r", encoding="utf-8") as f:
        datos = json.load(f)

    ids = [registro.get("id") for registro in datos]

    print("=== VALIDACIÓN DEL DATASET ===")
    print("Archivo:", archivo.name)
    print("Registros:", len(datos))
    print("IDs duplicados:", len(ids) - len(set(ids)))

    print(
        "Registros sin precio:",
        sum(not registro.get("precio_actual") for registro in datos)
    )

    print(
        "Registros sin ubicación:",
        sum(not registro.get("ubicacion") for registro in datos)
    )

    print(
        "Registros sin título:",
        sum(not registro.get("titulo") for registro in datos)
    )

=== VALIDACIÓN DEL DATASET ===
Archivo: portal_venta_departamento_santiago-metropolitana_20261022_041515_utc.json
Registros: 192
IDs duplicados: 0
Registros sin precio: 0
Registros sin ubicación: 0
Registros sin título: 0


In [2]:
from collections import Counter

def identificar_moneda(precio):

    precio = str(precio).strip().upper()

    if precio.startswith("UF"):
        return "UF"

    elif precio.startswith("US$") or precio.startswith("USD"):
        return "USD"

    elif precio.startswith("$") or precio.startswith("CLP"):
        return "CLP"

    else:
        return "No identificada"


# Clasificar los precios
monedas = [
    identificar_moneda(registro["precio_actual"])
    for registro in datos
]

# Contar publicaciones por moneda
conteo_monedas = Counter(monedas)

print("=== DISTRIBUCIÓN DE MONEDAS ===")

for moneda, cantidad in conteo_monedas.items():

    porcentaje = cantidad / len(datos) * 100

    print(
        f"{moneda}: {cantidad} publicaciones "
        f"({porcentaje:.2f}%)"
    )

=== DISTRIBUCIÓN DE MONEDAS ===
UF: 155 publicaciones (80.73%)
CLP: 37 publicaciones (19.27%)


In [3]:
import json
from pathlib import Path

carpeta = Path(
    r"C:\Users\felip\Documents\GitHub\Proyecto_inmobiliario"
) / "datalake_crudo" / "portal_inmobiliario"

# Buscar los archivos de casas
archivos_casas = list(
    carpeta.glob("portal_venta_casa_*.json")
)

if not archivos_casas:
    print("No se encontraron archivos de casas.")

else:
    # Seleccionar el archivo más reciente
    archivo_casas = max(
        archivos_casas,
        key=lambda ruta: ruta.stat().st_mtime
    )

    with open(archivo_casas, "r", encoding="utf-8") as f:
        datos_casas = json.load(f)

    ids = [registro.get("id") for registro in datos_casas]

    print("=== VALIDACIÓN DE CASAS ===")
    print("Archivo:", archivo_casas.name)
    print("Registros:", len(datos_casas))
    print("IDs duplicados:", len(ids) - len(set(ids)))
    print(
        "Sin precio:",
        sum(not p.get("precio_actual") for p in datos_casas)
    )
    print(
        "Sin ubicación:",
        sum(not p.get("ubicacion") for p in datos_casas)
    )

    print("\n=== PRIMERAS 5 PUBLICACIONES ===")

    for publicacion in datos_casas[:5]:
        print("-" * 40)
        print("Título:", publicacion["titulo"])
        print("Precio:", publicacion["precio_actual"])
        print("Ubicación:", publicacion["ubicacion"])
        print("Atributos:", publicacion["atributos"])

=== VALIDACIÓN DE CASAS ===
Archivo: portal_venta_casa_santiago-metropolitana_20261022_042539_utc.json
Registros: 48
IDs duplicados: 0
Sin precio: 0
Sin ubicación: 0

=== PRIMERAS 5 PUBLICACIONES ===
----------------------------------------
Título: Casa Comercial | 380 M² | 9+ Privados | Barrio Republica.
Precio: UF 10.500
Ubicación: Av. Ejército Libertador, Santiago, Ejército - Toesca, Santiago
Atributos: ['9 dormitorios', '6 baños', '264 m² útiles']
----------------------------------------
Título: Casa Antigua/ Funcionando/ 7 Dormitorios/8 Baño/rol Hotel
Precio: UF 5.900
Ubicación: La Fetra 177, Santiago, Barrio Brasil, Santiago
Atributos: ['7 dormitorios', '8 baños', '183 m² útiles']
----------------------------------------
Título: Casa 2 Pisos Barrio Patrimonial Yungay En Venta
Precio: UF 4.700
Ubicación: Compañía De Jesus 2828, Santiago, Barrio Yungay, Santiago
Atributos: ['6 dormitorios', '2 baños', '254 m² útiles']
----------------------------------------
Título: Casa En Venta E

In [4]:
# Identificadores de departamentos
ids_departamentos = {
    p["id"] for p in datos
}

# Identificadores de casas
ids_casas = {
    p["id"] for p in datos_casas
}

# Identificadores presentes en ambos datasets
ids_compartidos = ids_departamentos & ids_casas

# Total de identificadores diferentes
ids_unicos = ids_departamentos | ids_casas

print("=== VALIDACIÓN ENTRE CATEGORÍAS ===")
print("Departamentos:", len(ids_departamentos))
print("Casas:", len(ids_casas))
print("IDs compartidos:", len(ids_compartidos))
print("Total de anuncios únicos:", len(ids_unicos))

=== VALIDACIÓN ENTRE CATEGORÍAS ===
Departamentos: 192
Casas: 48
IDs compartidos: 0
Total de anuncios únicos: 240


In [5]:
import json
from pathlib import Path

carpeta = Path(
    r"C:\Users\felip\Documents\GitHub\Proyecto_inmobiliario"
) / "datalake_crudo" / "portal_inmobiliario"

# Archivos de las dos búsquedas
archivo_usadas = carpeta / (
    "portal_venta_casa_santiago-metropolitana_"
    "20261022_042539_utc.json"
)

archivo_general = carpeta / (
    "portal_venta_casa_santiago-metropolitana_"
    "20261022_044239_utc.json"
)

# Cargar ambos datasets
with open(archivo_usadas, "r", encoding="utf-8") as f:
    casas_usadas = json.load(f)

with open(archivo_general, "r", encoding="utf-8") as f:
    casas_general = json.load(f)

# Obtener identificadores
ids_usadas = {p["id"] for p in casas_usadas}
ids_general = {p["id"] for p in casas_general}

# Comparar conjuntos
compartidos = ids_usadas & ids_general
solo_usadas = ids_usadas - ids_general
solo_general = ids_general - ids_usadas

print("=== COMPARACIÓN DE BÚSQUEDAS ===")
print("Casas usadas:", len(ids_usadas))
print("Casas en general:", len(ids_general))
print("Anuncios compartidos:", len(compartidos))
print("Solo en búsqueda de usadas:", len(solo_usadas))
print("Solo en búsqueda general:", len(solo_general))
print("Total de anuncios únicos:", len(ids_usadas | ids_general))

=== COMPARACIÓN DE BÚSQUEDAS ===
Casas usadas: 48
Casas en general: 48
Anuncios compartidos: 48
Solo en búsqueda de usadas: 0
Solo en búsqueda general: 0
Total de anuncios únicos: 48
